# 04 · Descriptive training (train/val, resume, best-checkpoint)
Wires the loaders from notebook 02 into the `Trainer` and runs `fit()`.

**Features shown:** train/val split, **tqdm** progress, per-step JSONL metrics, **auto-resume** from `last.ckpt`, **best-only** checkpointing (saves only when val-loss improves), gradient clipping + NaN-skip guards, warmup→cosine LR, and **MPS** device selection.

In [ ]:
# --- Bootstrap: make the package importable without installing, and stay OFFLINE.
import os, sys
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


In [ ]:
from pathlib import Path
from gemma_ft_json.config import load_config
from gemma_ft_json.tokenization import build_tokenizer
from gemma_ft_json.data.transforms import build_image_transform
from gemma_ft_json.data.dataset import TableJsonDataset
from gemma_ft_json.data.collate import build_dataloaders
from gemma_ft_json.models.vlm import build_model
from gemma_ft_json.training.trainer import Trainer
from gemma_ft_json.utils.device import get_device, describe_device
from gemma_ft_json.utils.logging_utils import setup_logging
from gemma_ft_json.utils.seed import set_seed

In [ ]:
cfg = load_config(ROOT / 'configs' / 'default.yaml')
# --- small, fast, fully-offline demo settings (delete for real training) ---
cfg.model.vision.image_size = 128; cfg.model.vision.patch_size = 16
cfg.model.vision.embed_dim = 128; cfg.model.vision.depth = 3; cfg.model.vision.num_heads = 4
cfg.model.decoder.hidden_size = 192
cfg.data.max_target_tokens = 128
cfg.training.epochs = 2; cfg.training.grad_accum_steps = 1
cfg.training.log_every_steps = 1; cfg.training.eval_every_steps = 0
cfg.paths.runs_dir = str(ROOT / 'runs')
set_seed(cfg.project.seed)
device = get_device(cfg.project.device)   # mps on Apple Silicon, else cpu/cuda
print('device:', describe_device(device))

In [ ]:
man = ROOT / 'data' / 'demo' / 'manifests'
tokenizer = build_tokenizer(cfg.model.decoder)
transform = build_image_transform(cfg.model.vision)
train_ds = TableJsonDataset(man / 'train.jsonl', tokenizer, transform, cfg.data.prompt, cfg.data.max_target_tokens)
val_ds   = TableJsonDataset(man / 'val.jsonl',   tokenizer, transform, cfg.data.prompt, cfg.data.max_target_tokens)
train_loader, val_loader = build_dataloaders(train_ds, val_ds, pad_id=tokenizer.pad_id,
                                             batch_size=cfg.dataloader.batch_size, num_workers=0)
model, _ = build_model(cfg)
print('trainable params:', model.num_trainable(), '/', model.num_total())

In [ ]:
run_dir = Path(cfg.paths.runs_dir) / cfg.project.name
setup_logging(run_dir, level=cfg.logging.level, filename=cfg.logging.log_filename)
cfg.save_snapshot(run_dir / 'config.snapshot.yaml')   # reproducibility + inference
trainer = Trainer(cfg, model, train_loader, val_loader, device, run_dir)
trainer.fit()
print('best val loss:', trainer.best_val)
print('checkpoints:', [p.name for p in run_dir.glob('*.ckpt')])

**Resume:** re-running the cell above resumes automatically from `last.ckpt` (epoch, optimizer, scheduler, RNG all restored) because `training.resume=true`.

Metrics stream to `runs/<name>/metrics.jsonl` — open notebook **05** *while this runs* to watch the curves live.